# Notebook 01 — Strategy classifier validation

Re-validates the R8 LightGBM strategy classifier against the
operator's hand-labelled set (`strategy_labels`). Surfaces wallets
where the classifier disagrees with manual labels — they are the
primary candidates for the next labelling iteration.

_Spec_: `docs/ROUND_13_CALIBRATION_AND_RESEARCH.md` § 3.5


## Setup (re-uses 00_data_loader)


In [ ]:
%run 00_data_loader.ipynb


## Join predictions ↔ hand-labels


In [ ]:
import pandas as pd

try:
    con.execute("CREATE OR REPLACE VIEW strategy_labels AS SELECT * FROM read_parquet(str(COLD_DIR / 'strategy_labels' / '*.parquet'))")
except Exception:
    pass  # OK if absent — graceful empty

df = con.execute('''
    SELECT lsh.wallet_address,
           lsh.predicted_strategy,
           lsh.confidence,
           sl.primary_strategy AS labelled_strategy,
           sl.confidence       AS label_confidence
    FROM leader_strategy_history lsh
    LEFT JOIN strategy_labels sl
      ON lsh.wallet_address = sl.wallet_address
    WHERE sl.primary_strategy IS NOT NULL
''').fetchdf()

if df.empty:
    print('NO DATA — populate strategy_labels via the operator labelling sprint (R8 § 7.A).')
else:
    df['agrees'] = df['predicted_strategy'] == df['labelled_strategy']
    print(f'Labelled pairs: {len(df)}')
    print(f'Agreement rate: {df.agrees.mean():.1%}')
    display(df[~df.agrees].head(20))
